# Facial Keypoints Detection — v4b — Adaptive Wing Loss + Weighted Loss Map, 48x48 heatmap, CoordConv, DARK decoding

Adaptive Wing (Oregon State University, arXiv 1904.07399). Its ablation: MSE 5.39 -> Adaptive Wing 4.65 -> + Weighted Loss Map 4.30 NME on WFLW, both free at inference; pose and expression subsets, the closest analogue to our nose-tip and lower-lip errors, improve 14-16%. The paper never tests coarse heatmaps or soft-argmax, so the heatmap is raised to 48x48 and CoordConv (+2 channels) is added as in the paper. DARK is evaluated at test time on the same heatmaps against soft-argmax and argmax + quarter shift.

Same 88% / 12% split as v2 (`random_state=42`), so the held-out RMSE is directly comparable to v2's 2.4385px. One log line per epoch goes to `logs/v4b.log`; the best weights are checkpointed on every improvement to `checkpoints/v4b.pt`. No figures are drawn during training: every epoch appends statistics (loss split, held-out RMSE overall and per coordinate, centre-pull slope, softmax entropy, temperature, gradient norm, learning rate) to `results/v4b_history.json`; they are plotted in the post-training notebook.

In [1]:
import json, time
import torch
import fkd_v4_common as C
print('device:', C.device(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else '')
data = C.load_data('data')
print('train rows', len(data['idx_tr']), '| held-out rows', len(data['idx_va']))

device: cuda NVIDIA L4


train rows 6203 | held-out rows 846


In [2]:
RUN = 'v4b'
CFG = {'epochs': 240,
 'a_max': 40,
 'a_patience': 6,
 'batch': 512,
 'lr': 0.001,
 'lr_b': 0.001,
 'sched': 'step',
 'patience': 30,
 'seed': 42,
 'opt': 'adamw',
 'wd': 0.0001,
 'H': 48,
 'sigma': 2.0,
 'heatmap_loss': 'awing',
 'hm_frac': 0.2,
 'aug': {'affine': True, 'rot_deg': 30, 'scale': 0.2, 'translate': 0.06, 'photometric': True, 'cutout': True}}
model = C.SingleStageModel(H=48, n_deconv=2, coordconv=True, beta=15.0)
print(sum(p.numel() for p in model.parameters()), 'parameters')

1671952 parameters


## Training (Stage A heatmap warm-up to plateau, then Stage B) — statistics only, no figures

In [3]:
t0 = time.time()
model, history, best = C.train(RUN, model, data, CFG)
print(f'best held-out RMSE {best:.4f}px | wall {time.time()-t0:.0f}s')

2026-10-08T12:28:26Z [v4b] start | params=1,671,952 | cfg={"epochs": 240, "a_max": 40, "a_patience": 6, "batch": 512, "lr": 0.001, "lr_b": 0.001, "sched": "step", "patience": 30, "seed": 42, "opt": "adamw", "wd": 0.0001, "H": 48, "sigma": 2.0, "heatmap_loss": "awing", "hm_frac": 0.2, "aug": {"affine": true, "rot_deg": 30, "scale": 0.2, "translate": 0.06, "photometric": true, "cutout": true}}


fkd_v4_common.py:510: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /__w/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:820.)
  k = len(idx); s["loss"] += float(loss) * k; s["l1"] += float(l1) * k; s["hm"] += float(ht) * k; s["n"] += k


2026-10-08T12:29:05Z [v4b:A] epoch   0 train_hm 2.05955 val_hm 0.71967 val_rmse 20.7476 beta 15.00 35.4s


2026-10-08T12:29:16Z [v4b:A] epoch   1 train_hm 0.73858 val_hm 0.69730 val_rmse 16.0749 beta 15.00 10.6s


2026-10-08T12:29:29Z [v4b:A] epoch   2 train_hm 0.67571 val_hm 0.66266 val_rmse 14.6208 beta 15.00 12.9s


...

2026-10-08T12:52:09Z [v4b:B1] epoch 222 loss 1.4652 (l1 1.1382 hm 0.2073) val_rmse 1.8144 best 1.7986 slope 0.790 lr 1.0e-05 beta 21.23 gnorm 10.47 3.3s


2026-10-08T12:52:12Z [v4b:B1] epoch 223 loss 1.4647 (l1 1.1384 hm 0.2068) val_rmse 1.8135 best 1.7986 slope 0.790 lr 1.0e-05 beta 21.23 gnorm 7.77 3.3s


2026-10-08T12:52:12Z [v4b:B1] early stop at epoch 223 (no improvement for 30 epochs)


2026-10-08T12:52:12Z [v4b] done | best held-out RMSE 1.7986 px | epochs used 224


best held-out RMSE 1.7986px | wall 1427s


## Held-out evaluation: decoders x flip test, per-keypoint RMSE, centre-pull slopes

In [4]:
res = C.evaluate_and_save(RUN, model, data, CFG, decoders=('softargmax', 'argmax_shift', 'dark'))
for k, v in res.items():
    if k.startswith('rmse') or k in ('shrinkage_slope_mean', 'beta_final'):
        print(f'{k:28s} {v}')
print('v2 reference 2.4385 px | target 1.9 px')

rmse_softargmax              1.7985634803771973
rmse_softargmax_tta          1.74066162109375
rmse_argmax_shift            2.0473737716674805
rmse_argmax_shift_tta        1.9026604890823364
rmse_dark                    1.9364879131317139
rmse_dark_tta                1.824009656906128
shrinkage_slope_mean         0.7935321877455525
beta_final                   21.244365692138672
v2 reference 2.4385 px | target 1.9 px


## Test-set submission (written, not submitted)
Decoder chosen by held-out RMSE with flip test among the three evaluated above (a selection over three options, stated as such).

In [5]:
cands = {d: res[f'rmse_{d}_tta'] for d in ('softargmax', 'argmax_shift', 'dark')}
best_dec = min(cands, key=cands.get)
n = C.write_submission(RUN, model, data, CFG, tta=True, decoder=best_dec)
print('decoder', best_dec, '| held-out', round(cands[best_dec], 4), '| rows', n)

decoder softargmax | held-out 1.7407 | rows 27124
